# LightGBM — Street Hail Demand Forecasting

**Goal**: Predict trip count per zone per 15-min slot, beating all baselines from notebook 03.

Pipeline:
1. Load demand table
2. Feature engineering (temporal + lag + spatial)
3. Train LightGBM with Poisson objective (count data)
4. Evaluate vs baselines
5. Feature importance
6. Error analysis by zone and hour

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

ROOT         = Path('..')
DEMAND_PATH  = ROOT / 'data' / 'processed' / 'demand_enriched.parquet'  # cleaned + zero-filled + features
BASELINE_CSV = ROOT / 'data' / 'processed' / 'baseline_results.csv'
LOOKUP       = ROOT / 'Meta Data' / 'Lookups' / 'taxi_zone_lookup.csv'

## 1. Load Demand Table

In [ ]:
# All features pre-built by 02b — load and go, nothing to recompute
demand = pd.read_parquet(DEMAND_PATH)
demand['time_bucket'] = pd.to_datetime(demand['time_bucket'])

print(f'Rows  : {len(demand):,}')
print(f'Cols  : {list(demand.columns)}')
print(f'Range : {demand["time_bucket"].min().date()} → {demand["time_bucket"].max().date()}')
print(f'Zones : {demand["PULocationID"].nunique()}')

## 3. Train / Test Split

In [ ]:
cutoff = demand['time_bucket'].max() - pd.Timedelta(weeks=4)

# All features already in demand_enriched — just declare which ones to use
FEATURES = [
    'PULocationID',
    # Temporal — raw
    'hour', 'minute', 'dayofweek', 'is_weekend', 'month', 'dayofyear', 'weekofyear', 'year',
    'slot_of_day',
    # Temporal — cyclical
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    # Context flags
    'is_holiday', 'cbd_pricing_active', 'is_airport_zone',
    # Zone metadata
    'borough_id', 'service_zone_id',
    # Zone historical baseline
    'zone_slot_baseline',
    # Lags
    'lag_15min', 'lag_1h', 'lag_2h', 'lag_1day', 'lag_1week',
    # Rolling means
    'roll_mean_1h', 'roll_mean_2h', 'roll_mean_1day',
]
TARGET = 'trip_count'

# dropna only on rows where lag/roll features are NaN (first slots per zone — expected)
train = demand[demand['time_bucket'] <= cutoff].dropna(subset=FEATURES)
test  = demand[demand['time_bucket'] >  cutoff].dropna(subset=FEATURES)

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Train : {len(X_train):,} rows')
print(f'Test  : {len(X_test):,} rows')
print(f'Features: {len(FEATURES)}')

## 4. Train LightGBM

In [ ]:
params = {
    'objective':         'poisson',   # count data — better than MSE for trip counts
    'metric':            'rmse',
    'n_estimators':      1000,
    'learning_rate':     0.05,
    'num_leaves':        127,
    'min_child_samples': 50,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'lambda_l1':         0.1,
    'lambda_l2':         0.1,
    'verbose':           -1,
    'n_jobs':            -1,
    'random_state':      42,
}

model = lgb.LGBMRegressor(**params)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(100)],
)

print(f'\nBest iteration: {model.best_iteration_}')

## 5. Evaluate vs Baselines

In [ ]:
y_pred = model.predict(X_test)
y_pred = np.maximum(y_pred, 0)  # clip negatives

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mask = y_test > 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100

lgbm_result = pd.DataFrame([{'Model': 'LightGBM', 'MAE': round(mae,3), 'RMSE': round(rmse,3), 'MAPE (%)': round(mape,2)}])

# Load baseline results
baseline_results = pd.read_csv(BASELINE_CSV)
all_results = pd.concat([baseline_results, lgbm_result], ignore_index=True)
print(all_results.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MAPE (%)']
palette = ['#5b9bd5', '#ed7d31', '#a9d18e', '#ff4444']

for ax, metric in zip(axes, metrics):
    bars = ax.bar(all_results['Model'], all_results[metric], color=palette)
    # Highlight LightGBM bar
    bars[-1].set_edgecolor('black')
    bars[-1].set_linewidth(1.5)
    ax.set_title(metric)
    ax.set_xticklabels(all_results['Model'], rotation=20, ha='right', fontsize=8)

plt.suptitle('LightGBM vs Baselines', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
importance = pd.DataFrame({
    'feature':    FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=importance, x='importance', y='feature', palette='Blues_r', ax=ax)
ax.set_title('LightGBM Feature Importance (gain)', fontsize=12)
ax.set_xlabel('Importance')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 7. Error Analysis

In [ ]:
test_eval = test[['PULocationID', 'time_bucket', 'trip_count', 'hour', 'dayofweek']].copy()
test_eval['predicted'] = y_pred
test_eval['abs_error'] = np.abs(test_eval['trip_count'] - test_eval['predicted'])

# MAE by hour
mae_by_hour = test_eval.groupby('hour')['abs_error'].mean()

# MAE by day of week
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
mae_by_dow = test_eval.groupby('dayofweek')['abs_error'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(mae_by_hour.index, mae_by_hour.values, color='steelblue')
axes[0].set_title('MAE by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Mean Absolute Error')
axes[0].set_xticks(range(0,24))

axes[1].bar([day_names[i] for i in mae_by_dow.index], mae_by_dow.values, color='coral')
axes[1].set_title('MAE by Day of Week')
axes[1].set_ylabel('Mean Absolute Error')

plt.tight_layout()
plt.show()

In [ ]:
# Worst and best predicted zones
zones_lookup = pd.read_csv(LOOKUP)
mae_by_zone = (
    test_eval.groupby('PULocationID')['abs_error'].mean()
    .reset_index()
    .merge(zones_lookup[['LocationID','Zone','Borough']], left_on='PULocationID', right_on='LocationID')
    .sort_values('abs_error', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top10_worst = mae_by_zone.head(10)
axes[0].barh(top10_worst['Zone'], top10_worst['abs_error'], color='tomato')
axes[0].invert_yaxis()
axes[0].set_title('10 Hardest Zones to Predict')
axes[0].set_xlabel('MAE')

top10_best = mae_by_zone.tail(10).sort_values('abs_error')
axes[1].barh(top10_best['Zone'], top10_best['abs_error'], color='mediumseagreen')
axes[1].invert_yaxis()
axes[1].set_title('10 Easiest Zones to Predict')
axes[1].set_xlabel('MAE')

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs predicted — sample
sample = test_eval.sample(min(8000, len(test_eval)), random_state=42)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(sample['trip_count'], sample['predicted'], alpha=0.2, s=4, color='steelblue')
lim = max(sample['trip_count'].max(), sample['predicted'].max())
ax.plot([0, lim], [0, lim], 'r--', linewidth=1.2, label='Perfect prediction')
ax.set_xlabel('Actual trip count')
ax.set_ylabel('Predicted trip count')
ax.set_title('LightGBM: Actual vs Predicted')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save model
model.booster_.save_model(str(ROOT / 'data' / 'processed' / 'lgbm_demand_model.txt'))
all_results.to_csv(ROOT / 'data' / 'processed' / 'all_model_results.csv', index=False)
print('Model and results saved.')
print('\n=== Final Comparison ===')
print(all_results.to_string(index=False))